# AdaptRoute — Gating Network Training (LoRA)

This notebook trains a routing classifier to direct queries to specialized LoRA adapters.

**Key Features:**
- **LoRA on DistilBERT** — trains low-rank adapter matrices to maintain efficiency.
- **Class-balanced training** — ensures equal representation for all task categories.

| ID | Label | Role |
|----|-------|------|
| 0 | `code` | Route to `lora-code` |
| 1 | `math` | Route to `lora-math` |
| 2 | `qa` | Route to `lora-qa` |
| 3 | `medical` | Route to `lora-medical` |

**Expected runtime on A100:** ~10 minutes

## 1. Configuration
Only change `HF_USERNAME`. Everything else is tuned for the known dataset sizes.

In [35]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
HF_USERNAME   = "kunjcr2"
HF_REPO_NAME  = "gating-bert-adaptroute"
WANDB_PROJECT = "adaptroute-gate"

# Removed mal.json
DATA_FILES = ["code.json", "math.json", "qa.json", "medical.json"]

MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH       = 128

LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.1
LORA_TARGET_MODULES = ["q_lin", "k_lin", "v_lin"]

BATCH_SIZE      = 32
NUM_EPOCHS      = 5
LEARNING_RATE   = 2e-4
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.06
VAL_SPLIT       = 0.1
TEST_SPLIT      = 0.1
SEED            = 42
OUTPUT_DIR      = "./gate-checkpoint"
# ──────────────────────────────────────────────────────────────────────────────

# Updated to 4 classes
LABEL2ID = {"code": 0, "math": 1, "qa": 2, "medical": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
FULL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"Repo   : {FULL_REPO_ID}")
print(f"Labels : {LABEL2ID}")

Repo   : kunjcr2/gating-bert-adaptroute
Labels : {'code': 0, 'math': 1, 'qa': 2, 'medical': 3}


## 2. Authentication

In [36]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("✓ HuggingFace login OK")

✓ HuggingFace login OK


In [37]:
if WANDB_PROJECT:
    import wandb
    wandb.login(key=userdata.get("WANDB_API"))
    wandb.init(project=WANDB_PROJECT, name="gate-distilbert-lora")
    print("✓ W&B initialised")
else:
    print("Skipping W&B")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


eval/accuracy,▄▆▆▅█▂▁▅▆▅
eval/injection_fpr,▁▁▁▁▁▁█▁▁▃
eval/injection_recall,▁▆▆▃█▁▆▃▆▆
eval/loss,▆▂▁▃▁█▁▄▁▂
eval/macro_f1,▄▆▆▅█▂▁▅▆▅
eval/runtime,█▁▃▄▂▆▆█▇█
eval/samples_per_second,▁█▆▅▇▃▃▁▂▁
eval/steps_per_second,▁█▆▅▇▃▃▁▂▁
test/accuracy,▁█
test/injection_fpr,█▁
+11,...


✓ W&B initialised


## 3. Load & Balance Data

**Root cause of the FPR problem:** if `mal.json` has fewer samples than the others,
the model sees proportionally more injection text, which has very distinctive lexical
patterns ('ignore', 'forget', 'as DAN', etc.). Oversampling benign classes to match
injection count AND undersampling injection to match benign count balances this.

We also compute **class weights** for the loss function — benign classes get higher
weight so a wrong prediction on a real query hurts more than a missed injection.

In [38]:
import json
import random
import numpy as np
from collections import Counter

random.seed(SEED)
np.random.seed(SEED)


def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    texts, labels = [], []
    for record in data:
        text  = str(record["user"]).strip()
        label = int(record["label"])
        if text:
            texts.append(text)
            labels.append(label)
    return texts, labels


all_texts, all_labels = [], []
for path in DATA_FILES:
    texts, labels = load_json_file(path)
    all_texts  += texts
    all_labels += labels
    print(f"  {path:15s} → {len(texts):,} samples  (label {labels[0]} / '{ID2LABEL[labels[0]]}')")

raw_counts = Counter(all_labels)
print(f"\nRaw class dist : { {ID2LABEL[k]: v for k, v in sorted(raw_counts.items())} }")

  code.json       → 7,000 samples  (label 0 / 'code')
  math.json       → 7,500 samples  (label 1 / 'math')
  qa.json         → 10,000 samples  (label 2 / 'qa')
  medical.json    → 10,000 samples  (label 3 / 'medical')

Raw class dist : {'code': 7000, 'math': 7500, 'qa': 10000, 'medical': 10000}


In [39]:
# ── Balance: target median count for all benign classes ──
by_class = {}
for text, label in zip(all_texts, all_labels):
    by_class.setdefault(label, []).append(text)

class_ids   = list(LABEL2ID.values())
class_sizes = [len(by_class[lid]) for lid in class_ids]
target_n     = int(np.median(class_sizes))

balanced_texts, balanced_labels = [], []
for label_id, texts in by_class.items():
    if len(texts) >= target_n:
        sampled = random.sample(texts, target_n)
    else:
        sampled = texts + random.choices(texts, k=target_n - len(texts))
    balanced_texts  += sampled
    balanced_labels += [label_id] * len(sampled)

balanced_counts = Counter(balanced_labels)
print(f"Target per class : {target_n:,}")
print(f"Balanced dist    : { {ID2LABEL[k]: v for k, v in sorted(balanced_counts.items())} }")
print(f"Total samples    : {len(balanced_texts):,}")

Target per class : 8,750
Balanced dist    : {'code': 8750, 'math': 8750, 'qa': 8750, 'medical': 8750}
Total samples    : 35,000


In [40]:
from sklearn.model_selection import train_test_split

train_texts, train_labels = [], []
val_texts,   val_labels   = [], []
test_texts,  test_labels  = [], []

for label_id in sorted(set(balanced_labels)):
    class_texts = [t for t, l in zip(balanced_texts, balanced_labels) if l == label_id]
    tr_val, te  = train_test_split(class_texts, test_size=TEST_SPLIT, random_state=SEED)
    val_frac    = VAL_SPLIT / (1 - TEST_SPLIT)
    tr, va      = train_test_split(tr_val, test_size=val_frac, random_state=SEED)
    train_texts += tr;  train_labels += [label_id] * len(tr)
    val_texts   += va;  val_labels   += [label_id] * len(va)
    test_texts  += te;  test_labels  += [label_id] * len(te)

combined = list(zip(train_texts, train_labels))
random.shuffle(combined)
train_texts, train_labels = map(list, zip(*combined))

print(f"Train : {len(train_texts):,}")
print(f"Val   : {len(val_texts):,}")
print(f"Test  : {len(test_texts):,}")

Train : 27,996
Val   : 3,504
Test  : 3,500


In [41]:
import torch
from sklearn.utils.class_weight import compute_class_weight

# Standard balanced weights for the 4 routing classes
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(LABEL2ID.values())),
    y=np.array(train_labels),
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {ID2LABEL[i]:12s}: {w:.4f}")

Class weights:
  code        : 1.0000
  math        : 1.0000
  qa          : 1.0000
  medical     : 1.0000


## 4. Tokenise

In [42]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def make_hf_dataset(texts, labels):
    ds = Dataset.from_dict({"text": texts, "label": labels})
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True, padding="max_length", max_length=MAX_LENGTH,
        )
    ds = ds.map(tokenize, batched=True, batch_size=512)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")
    return ds

train_ds = make_hf_dataset(train_texts, train_labels)
val_ds   = make_hf_dataset(val_texts,   val_labels)
test_ds  = make_hf_dataset(test_texts,  test_labels)

print("Train:", train_ds)
print("Val:  ", val_ds)

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3504 [00:00<?, ? examples/s]

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 27996
})
Val:   Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3504
})


## 5. Model + LoRA

LoRA is applied to the Q, K, V projection matrices in every DistilBERT attention
layer (`q_lin`, `k_lin`, `v_lin`). The base model weights are **frozen** — only the
small adapter matrices (~1% of params) are trained. This prevents the full model from
memorising the injection vocabulary pattern across all its weights.

In [43]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_config = LoraConfig(
    task_type        = TaskType.SEQ_CLS,
    r                = LORA_R,
    lora_alpha       = LORA_ALPHA,
    lora_dropout     = LORA_DROPOUT,
    target_modules   = LORA_TARGET_MODULES,
    bias             = "none",
    modules_to_save  = ["classifier", "pre_classifier"],  # keep classifier head trainable
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expect ~1-3% of total params to be trainable — that's correct for LoRA on DistilBERT

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,036,036 || all params: 67,992,584 || trainable%: 1.5237


## 6. Custom Trainer with Weighted Loss

Subclass `Trainer` to inject class-weighted cross-entropy. This is the second
defence against injection overfit: wrong predictions on benign classes cost more.

In [44]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        weights = class_weights_tensor.to(logits.device)
        loss    = nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="macro")

    return {
        "accuracy" : round(float(acc), 4),
        "macro_f1" : round(float(f1), 4),
    }

## 7. Training

In [45]:
from transformers import TrainingArguments

total_steps  = (len(train_ds) // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_steps                = warmup_steps,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,            # <--- This ensures the best model is loaded after training
    metric_for_best_model       = "accuracy",      # <--- We define 'best' as highest accuracy
    greater_is_better           = True,
    fp16                        = True,
    logging_steps               = 1,
    report_to                   = "wandb" if WANDB_PROJECT else "none",
    push_to_hub                 = False,
    seed                        = SEED,
)

trainer = WeightedLossTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")

Total steps  : 4370
Warmup steps : 262


In [46]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.000413,0.002184,0.999400,0.999400
2,0.000008,0.000883,0.999400,0.999400
3,0.000004,0.001103,0.999700,0.999700
4,0.000005,0.000830,0.999700,0.999700
5,0.000003,0.000815,0.999700,0.999700


TrainOutput(global_step=4375, training_loss=0.024437749850420472, metrics={'train_runtime': 146.3231, 'train_samples_per_second': 956.65, 'train_steps_per_second': 29.9, 'total_flos': 4747240635310080.0, 'train_loss': 0.024437749850420472, 'epoch': 5.0})

## 8. Evaluate on Test Set

**Key numbers to check:**
- `accuracy` > 90% overall

In [47]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

preds_out   = trainer.predict(test_ds)
pred_ids    = np.argmax(preds_out.predictions, axis=-1)
true_ids    = preds_out.label_ids
label_names = [ID2LABEL[i] for i in range(len(ID2LABEL))]

print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)
print(classification_report(true_ids, pred_ids, target_names=label_names))

cm    = confusion_matrix(true_ids, pred_ids)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
print("Confusion Matrix:")
print(cm_df.to_string())

FINAL TEST SET RESULTS
              precision    recall  f1-score   support

        code       1.00      1.00      1.00       875
        math       1.00      1.00      1.00       875
          qa       1.00      1.00      1.00       875
     medical       1.00      1.00      1.00       875

    accuracy                           1.00      3500
   macro avg       1.00      1.00      1.00      3500
weighted avg       1.00      1.00      1.00      3500

Confusion Matrix:
         code  math   qa  medical
code      874     1    0        0
math        0   875    0        0
qa          0     1  874        0
medical     0     0    0      875


In [48]:
import time

model.eval()
cpu_model = model.to("cpu")

sample = "Write a Python function that reverses a linked list."
enc = tokenizer(sample, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)

for _ in range(5):
    with torch.no_grad(): _ = cpu_model(**enc)

N = 100
t0 = time.perf_counter()
for _ in range(N):
    with torch.no_grad(): out = cpu_model(**enc)
ms = (time.perf_counter() - t0) / N * 1000

probs = torch.softmax(out.logits, dim=-1)[0]
print(f"Pred  : {ID2LABEL[probs.argmax().item()]}")
print(f"Probs : { {ID2LABEL[i]: round(probs[i].item(), 3) for i in range(len(ID2LABEL))} }")
print(f"CPU   : {ms:.2f} ms  (target < 10 ms)")

Pred  : code
Probs : {'code': 1.0, 'math': 0.0, 'qa': 0.0, 'medical': 0.0}
CPU   : 16.48 ms  (target < 10 ms)


## 9. Inference Helper

Drop this into your AdaptRoute pipeline unchanged.
Includes a **confidence threshold for hard routing**: if one adapter scores > 0.75,
don't soft-merge — just use that one. Avoids specialisation dilution on clear queries.

In [49]:
ADAPTER_LABELS = [l for l in LABEL2ID if l != "injection"]
HARD_ROUTE_CONFIDENCE = 0.75

def gate(
    query: str,
    top_k: int = 2,
) -> dict:
    """
    Simplified gate function focusing only on routing classifications.
    """
    model.eval()
    enc = tokenizer(
        query, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        logits = model(**enc).logits

    probs = torch.softmax(logits, dim=-1)[0]
    # Filter out the injection label from the results
    prob_dict = {ID2LABEL[i]: round(probs[i].item(), 4) for i in range(len(ID2LABEL)) if ID2LABEL[i] != "injection"}

    sorted_adapters = sorted(prob_dict, key=prob_dict.get, reverse=True)
    best_label = sorted_adapters[0]

    # Hard routing when one class dominates
    if prob_dict[best_label] >= HARD_ROUTE_CONFIDENCE:
        return {
            "top_adapters": [f"lora-{best_label}"],
            "top_weights": [1.0],
            "hard_routed": True,
            "probs": prob_dict,
        }

    # Soft routing: top-k with renormalised weights
    top_adapters = sorted_adapters[:top_k]
    raw_weights = [prob_dict[a] for a in top_adapters]
    weight_sum = sum(raw_weights)
    top_weights = [w / weight_sum for w in raw_weights] if weight_sum > 0 else [0.0] * len(raw_weights)

    return {
        "top_adapters": [f"lora-{a}" for a in top_adapters],
        "top_weights": top_weights,
        "hard_routed": False,
        "probs": prob_dict,
    }

# ── Updated Smoke test (No Injection) ──────────────────────────────────
smoke_queries = [
    "Write a Python function that sorts a list using quicksort.",
    "Solve the integral of x^2 from 0 to 1.",
    "What are the symptoms of Type 2 diabetes?",
    "What is the capital of France?",
    "Can you explain how transformers work in NLP?",
    "What medications are used to treat hypertension?",
]

print(f"{'Query':<55}  Result")
print("-" * 85)
for q in smoke_queries:
    r = gate(q)
    if r["hard_routed"]:
        result = f"HARD → {r['top_adapters'][0]}"
    else:
        pairs = list(zip(r["top_adapters"], [round(w, 2) for w in r["top_weights"]]))
        result = f"SOFT → {pairs}"
    print(f"{q[:53]:<55}  {result}")

Query                                                    Result
-------------------------------------------------------------------------------------
Write a Python function that sorts a list using quick    HARD → lora-code
Solve the integral of x^2 from 0 to 1.                   HARD → lora-math
What are the symptoms of Type 2 diabetes?                HARD → lora-medical
What is the capital of France?                           HARD → lora-qa
Can you explain how transformers work in NLP?            HARD → lora-medical
What medications are used to treat hypertension?         HARD → lora-medical


### Comprehensive Evaluation with Extended Smoke Queries
This cell tests the gating network against a broader variety of inputs to verify accuracy across all 4 target classes.

In [51]:
# Define a larger set of test cases with expected labels
extended_evaluation_set = [
    # --- CODE ---
    ("How do I implement a binary search tree in C++?", "code"),
    ("Write a regex to validate an email address.", "code"),
    ("Explain the difference between a list and a tuple in Python.", "code"),
    ("What is the time complexity of merge sort?", "code"),
    ("Create a React component for a navigation bar.", "code"),
    # --- MATH ---
    ("What is the derivative of sin(x) * cos(x)?", "math"),
    ("Solve for x: 2x^2 - 5x + 3 = 0.", "math"),
    ("Calculate the volume of a sphere with radius 5.", "math"),
    ("What is the sum of the first 100 prime numbers?", "math"),
    ("Prove that there are infinitely many prime numbers.", "math"),
    # --- MEDICAL ---
    ("What are the long-term side effects of ibuprofen?", "medical"),
    ("How do I manage symptoms of seasonal allergies?", "medical"),
    ("What is the difference between a virus and a bacteria?", "medical"),
    ("Explain the stages of hypertension.", "medical"),
    ("What are the early warning signs of a stroke?", "medical"),
    # --- QA ---
    ("Who was the first woman to win a Nobel Prize?", "qa"),
    ("What is the population of Tokyo in 2024?", "qa"),
    ("Explain the historical significance of the Magna Carta.", "qa"),
    ("What is the tallest mountain in the solar system?", "qa"),
    ("How many countries are members of the United Nations?", "qa")
]

extended_evaluation_set += [
    # --- CODE ---
    ("Implement Dijkstra’s algorithm in Python.", "code"),
    ("What is the difference between REST and GraphQL APIs?", "code"),
    ("Write a SQL query to find the second highest salary in a table.", "code"),
    ("Explain how garbage collection works in Java.", "code"),
    ("Build a simple HTTP server using Node.js.", "code"),

    # --- MATH ---
    ("Find the integral of x^2 * e^x.", "math"),
    ("Solve the system of equations: x + y = 5, x - y = 1.", "math"),
    ("What is the limit of (1 + 1/n)^n as n approaches infinity?", "math"),
    ("Find the eigenvalues of a 2x2 matrix [[2,1],[1,2]].", "math"),
    ("What is the Taylor series expansion of e^x?", "math"),

    # --- MEDICAL ---
    ("What causes type 2 diabetes?", "medical"),
    ("How is hypertension typically treated?", "medical"),
    ("What are the symptoms of iron deficiency anemia?", "medical"),
    ("Explain how vaccines work in the human body.", "medical"),
    ("What are risk factors for heart disease?", "medical"),

    # --- QA ---
    ("Who developed the theory of general relativity?", "qa"),
    ("What is the capital of Australia?", "qa"),
    ("When did World War II end?", "qa"),
    ("What is the chemical symbol for gold?", "qa"),
    ("Which planet has the most moons?", "qa")
]

correct_count = 0
total_count = len(extended_evaluation_set)

print(f"{'Query':<60} | {'Expected':<10} | {'Predicted':<10} | {'Status'}")
print("-" * 100)

for query, expected_label in extended_evaluation_set:
    result = gate(query)
    # top_adapters returns e.g., ['lora-code']
    predicted_label = result['top_adapters'][0].replace('lora-', '')

    is_correct = predicted_label == expected_label
    if is_correct:
        correct_count += 1
        status = "✅"
    else:
        status = "❌"

    print(f"{query[:58]:<60} | {expected_label:<10} | {predicted_label:<10} | {status}")

accuracy = (correct_count / total_count) * 100
print("-" * 100)
print(f"Final Accuracy on Smoke Queries: {accuracy:.2f}% ({correct_count}/{total_count})")

Query                                                        | Expected   | Predicted  | Status
----------------------------------------------------------------------------------------------------
How do I implement a binary search tree in C++?              | code       | code       | ✅
Write a regex to validate an email address.                  | code       | code       | ✅
Explain the difference between a list and a tuple in Pytho   | code       | code       | ✅
What is the time complexity of merge sort?                   | code       | math       | ❌
Create a React component for a navigation bar.               | code       | code       | ✅
What is the derivative of sin(x) * cos(x)?                   | math       | math       | ✅
Solve for x: 2x^2 - 5x + 3 = 0.                              | math       | math       | ✅
Calculate the volume of a sphere with radius 5.              | math       | math       | ✅
What is the sum of the first 100 prime numbers?              | math       |

## 10. Save & Push to HuggingFace Hub

We merge the LoRA weights back into the base model before pushing so the
hub model is a single self-contained checkpoint (no PEFT dependency at inference).

In [54]:
from huggingface_hub import HfApi
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

merged_model = model.merge_and_unload()
merged_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model_card = f"""---
language: en
license: apache-2.0
tags:
  - text-classification
  - distilbert
  - adaptroute
  - router
  - lora
pipeline_tag: text-classification
---

# gating-bert-adaptroute

A 4-class DistilBERT classifier acting as the gating network for AdaptRoute.

## Labels
| ID | Label |
|----|-------|
| 0 | code |
| 1 | math |
| 2 | qa |
| 3 | medical |

## Architecture
- Base: `distilbert-base-uncased` (LoRA merged)
- Training: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}"""

with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(model_card)

api = HfApi()
api.create_repo(repo_id=FULL_REPO_ID, exist_ok=True, token=HF_TOKEN)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=FULL_REPO_ID, token=HF_TOKEN)
print(f"✓ Pushed to https://huggingface.co/{FULL_REPO_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...checkpoint-1750/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

  ...nt-1750/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

...

  ...ckpoint-1094/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...kpoint-1094/rng_state.pth: 100%|##########| 14.6kB / 14.6kB            

  ...ckpoint-3282/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...kpoint-2625/rng_state.pth: 100%|##########| 14.6kB / 14.6kB            

✓ Pushed to https://huggingface.co/kunjcr2/gating-bert-adaptroute


## 11. Verify — Reload from Hub

In [58]:
from transformers import pipeline as hf_pipeline

gate_pipe = hf_pipeline("text-classification", model=FULL_REPO_ID, top_k=None)

for q, _ in extended_evaluation_set:
    res = gate_pipe(q)
    top = sorted(res[0], key=lambda x: x["score"], reverse=True)[0]
    print(f"{q[:55]:<57} → {top['label']:12s} ({top['score']:.3f})")

print(f"\n✓ Reloaded from https://huggingface.co/{FULL_REPO_ID}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


How do I implement a binary search tree in C++?           → code         (0.934)
Write a regex to validate an email address.               → code         (0.999)
Explain the difference between a list and a tuple in Py   → code         (0.989)
What is the time complexity of merge sort?                → math         (0.885)
Create a React component for a navigation bar.            → code         (0.999)
What is the derivative of sin(x) * cos(x)?                → math         (0.999)
Solve for x: 2x^2 - 5x + 3 = 0.                           → math         (1.000)
Calculate the volume of a sphere with radius 5.           → math         (0.983)
What is the sum of the first 100 prime numbers?           → math         (1.000)
Prove that there are infinitely many prime numbers.       → math         (0.994)
What are the long-term side effects of ibuprofen?         → medical      (1.000)
How do I manage symptoms of seasonal allergies?           → medical      (1.000)
What is the difference betwe

## 12. Finish

In [59]:
if WANDB_PROJECT:
    wandb.finish()

print("Done.")
print(f"  Hub   : https://huggingface.co/{FULL_REPO_ID}")
print(f"  Local : {OUTPUT_DIR}/")

eval/accuracy,▁▁███
eval/loss,█▁▂▁▁
eval/macro_f1,▁▁███
eval/runtime,▂█▁▁▂
eval/samples_per_second,▇▁██▇
eval/steps_per_second,▇▁██▇
test/accuracy,▁
test/loss,▁
test/macro_f1,▁
test/runtime,▁
+7,...


Done.
  Hub   : https://huggingface.co/kunjcr2/gating-bert-adaptroute
  Local : ./gate-checkpoint/
